In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nexon4ik/shiza65656/shiza.csv


In [2]:
# 1. Установка (один раз)
!pip install -q transformers accelerate bitsandbytes scikit-learn

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import gc, time, numpy as np, pandas as pd
from tqdm import tqdm
import torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.7 MB/s eta 0:00:00:00:0100:01


In [3]:
# ========== Конфигурация ==========
MODEL_NAME = "ai-sage/GigaChat3-10B-A1.8B-bf16"
INPUT_CSV = "../data/public/shiza.csv"
OUTPUT_DIR = "../model/"
PROBE_LAYERS = [0,5,10,15,20,25]
NUM_ROWS = 5000               # сколько строк исходного датасета использовать
CHUNK_SIZE = 25              # обрабатывать по 25 строк за раз
CHECKPOINT_PATH = OUTPUT_DIR + "checkpoint.pkl"

# ========== Загрузка данных ==========
df = pd.read_csv(INPUT_CSV)
df_sample = df.head(NUM_ROWS).reset_index(drop=True)
print(f"Всего строк: {len(df_sample)} → примеров: {len(df_sample)*2}")

# ========== Загрузка модели (8-bit) ==========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,  # разрешаем offload на CPU
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",           # авто-распределение (с offload)
    trust_remote_code=True,
    torch_dtype=torch.float16,
    offload_folder="offload",    # папка для временного offload
)
model.eval()

Всего строк: 500 → примеров: 1000


config.json: 0.00B [00:00, ?B/s]

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

...

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

DeepseekV3ForCausalLM LOAD REPORT from: ai-sage/GigaChat3-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.enorm.weight                        | UNEXPECTED |  | 
model.layers.26.hnorm.weight                        | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_layernorm.weight     | UNEXPECTED |  | 
model.layers.26.mlp.experts.gate_up_proj            | UNEXPECTED |  | 
model.layers.26.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.26.eh_proj.weight                      | UNEXPECTED |  | 
model.layers.26.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_proj_with_mqa.weight | UNEXPECTED |  | 
model.layers.26.mlp.gate.weight                     | UNEXPECTED |  | 
model.layers.26.post_attention_layernorm.weight     | UNEXPECTED |  | 
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
mode

generation_config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

DeepseekV3ForCausalLM(
  (model): DeepseekV3Model(
    (embed_tokens): Embedding(128256, 1536)
    (layers): ModuleList(
      (0): DeepseekV3DecoderLayer(
        (self_attn): DeepseekV3Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=6144, bias=False)
          (kv_a_proj_with_mqa): Linear8bitLt(in_features=1536, out_features=576, bias=False)
          (kv_a_layernorm): DeepseekV3RMSNorm((512,), eps=1e-06)
          (kv_b_proj): Linear8bitLt(in_features=512, out_features=10240, bias=False)
          (o_proj): Linear8bitLt(in_features=6144, out_features=1536, bias=False)
        )
        (mlp): DeepseekV3MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): DeepseekV3RMSNorm((1536,), e

In [4]:
# ========== FeatureAccumulator (как в baseline) ==========
class FeatureAccumulator:
    def __init__(self, model, probe_layers):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks = []
        self._hidden = {}
    def attach(self):
        self._hidden.clear()
        layers = self.model.model.layers
        for idx in self.probe_layers:
            if idx >= len(layers): continue
            name = f"layer_{idx}"
            def make_hook(n):
                def hook(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return hook
            self._hooks.append(layers[idx].register_forward_hook(make_hook(name)))
    def detach(self):
        for h in self._hooks: h.remove()
        self._hooks.clear()
    def __enter__(self):
        self.attach()
        return self
    def __exit__(self, *args):
        self.detach()
    def compute_features(self, logits, input_ids, ans_start):
        seq_len = input_ids.shape[1]
        ans_len = seq_len - ans_start
    
        # --- Uncertainty features ---
        answer_logits = logits[0, ans_start-1:seq_len-1, :].float()
        answer_ids = input_ids[0, ans_start:seq_len]
        # ПЕРЕНЕСЁМ answer_ids на то же устройство, что и answer_logits
        answer_ids = answer_ids.to(answer_logits.device)
    
        log_probs = F.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)
    
        probs = F.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)
    
        uncertainty = np.array([
            token_lp.mean().item(), token_lp.min().item(), token_lp.max().item(),
            token_lp.std().item() if ans_len > 1 else 0.0,
            entropy.mean().item(), entropy.max().item(),
            entropy.std().item() if ans_len > 1 else 0.0,
            torch.exp(-token_lp.mean()).item(), float(ans_len), token_lp[0].item(),
            top1.mean().item(), top5.mean().item()
        ], dtype=np.float32)
    
        # --- Internal features & probe vector ---
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vec = last_hs[ans_start-1].cpu().float().numpy()
    
        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[ans_start-1].norm().item())
            int_scalars.append(hs[ans_start:seq_len].norm(dim=-1).mean().item())
        
            ans_hs = hs[ans_start-1:seq_len-1].unsqueeze(0)
            with torch.no_grad():
                norm_hs = self.model.model.norm(ans_hs)
                ll = self.model.lm_head(norm_hs).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p + 1e-10)).sum(dim=-1)
            int_scalars.append(ll_e.mean().item())
    
        first_e, last_e = int_scalars[2], int_scalars[-1]
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e / (first_e + 1e-10))
    
        internal = np.array(int_scalars, dtype=np.float32)
        self._hidden.clear()
        return {"uncertainty": uncertainty, "internal_scalars": internal, "probe_vec": probe_vec}

In [5]:
# ========== Препроцессинг (как в baseline) ==========
def preprocess(tokenizer, prompt, answer):
    prompt_enc = tokenizer.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True, tokenize=True)
    ans_start = len(prompt_enc["input_ids"])
    full_enc = tokenizer.apply_chat_template([{"role":"user","content":prompt},{"role":"assistant","content":answer}], tokenize=True)
    tok_ids = torch.tensor([full_enc["input_ids"]], dtype=torch.long)
    return tok_ids, ans_start

# ========== Восстановление чекпоинта ==========
start_idx = 0
all_unc = []; all_int = []; all_probe = []; all_labels = []
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'rb') as f:
        start_idx, all_unc, all_int, all_probe, all_labels = pickle.load(f)
    print(f"Восстановлен чекпоинт: обработано {start_idx} строк")
else:
    print("Новый запуск")

# ========== Обработка чанками ==========
accumulator = FeatureAccumulator(model, PROBE_LAYERS)

for chunk_start in range(start_idx, len(df_sample), CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, len(df_sample))
    print(f"\nОбработка строк {chunk_start}–{chunk_end-1}...")
    
    for idx in tqdm(range(chunk_start, chunk_end), desc="Внутри чанка"):
        row = df_sample.iloc[idx]
        prompt = f"Вопрос: {row['question']}\n\nОтвет:"
        
        # Правильный ответ (label 0)
        tok, start = preprocess(tokenizer, prompt, row['right_answer'])
        with accumulator, torch.no_grad():
            out = model(tok.to(model.device))
        f = accumulator.compute_features(out.logits, tok, start)
        all_unc.append(f["uncertainty"])
        all_int.append(f["internal_scalars"])
        all_probe.append(f["probe_vec"])
        all_labels.append(0)
        del out, tok
        torch.cuda.empty_cache()
        
        # Галлюцинация (label 1)
        tok, start = preprocess(tokenizer, prompt, row['hallucinated_answer'])
        with accumulator, torch.no_grad():
            out = model(tok.to(model.device))
        f = accumulator.compute_features(out.logits, tok, start)
        all_unc.append(f["uncertainty"])
        all_int.append(f["internal_scalars"])
        all_probe.append(f["probe_vec"])
        all_labels.append(1)
        del out, tok
        torch.cuda.empty_cache()
    
    # Сохраняем чекпоинт после чанка
    with open(CHECKPOINT_PATH, 'wb') as f:
        pickle.dump((chunk_end, all_unc, all_int, all_probe, all_labels), f)
    print(f"Чекпоинт сохранён до строки {chunk_end}")
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Собрано примеров: {len(all_labels)}")

# ========== Преобразование в массивы ==========
print("Преобразование в массивы...")
unc = np.stack(all_unc)           # (n_samples, 12)
ints = np.stack(all_int)          # (n_samples, N_int) 
probe = np.stack(all_probe)       # (n_samples, hidden_dim)
y = np.array(all_labels)          # (n_samples,)

# ========== РАЗДЕЛЕНИЕ НА TRAIN/TEST ДО PCA И SCALING ==========

X_raw = np.hstack([probe, ints, unc])   # объединяем все признаки без PCA
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# ========== PCA только на TRAIN ==========
print("PCA (64 компоненты) на train...")
pca = PCA(n_components=64, random_state=42)
probe_pca_train = pca.fit_transform(X_train[:, :probe.shape[1]])  # только probe-часть
probe_pca_test = pca.transform(X_test[:, :probe.shape[1]])

# Заменяем probe-колонки в train/test на PCA-версии
X_train_pca = np.hstack([probe_pca_train, X_train[:, probe.shape[1]:]])
X_test_pca = np.hstack([probe_pca_test, X_test[:, probe.shape[1]:]])

# ========== StandardScaler только на TRAIN ==========
print("Масштабирование на train...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled = scaler.transform(X_test_pca)

# ========== Сохранение ==========
# Сохраняем train/test отдельно
np.savez_compressed(OUTPUT_DIR + "features_train.npz", X=X_train_scaled, y=y_train)
np.savez_compressed(OUTPUT_DIR + "features_test.npz", X=X_test_scaled, y=y_test)

# Сохраняем трансформеры для возможного использования в будущем
with open(OUTPUT_DIR + "pca.pkl", "wb") as f: 
    pickle.dump(pca, f)
with open(OUTPUT_DIR + "scaler.pkl", "wb") as f: 
    pickle.dump(scaler, f)

print(f"\n🎉 Готово! Файлы сохранены в {OUTPUT_DIR}")
print(f"Train X: {X_train_scaled.shape}, y: {np.bincount(y_train)}")
print(f"Test X: {X_test_scaled.shape}, y: {np.bincount(y_test)}")

# Удаляем чекпоинт
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

Новый запуск

Обработка строк 0–24...



Внутри чанка: 100%|██████████| 25/25 [00:28<00:00,  1.15s/it]


Чекпоинт сохранён до строки 25

Обработка строк 25–49...



Внутри чанка: 100%|██████████| 25/25 [00:27<00:00,  1.10s/it]


Чекпоинт сохранён до строки 50

Обработка строк 50–74...


...

Чекпоинт сохранён до строки 450

Обработка строк 450–474...



Внутри чанка: 100%|██████████| 25/25 [00:28<00:00,  1.13s/it]


Чекпоинт сохранён до строки 475

Обработка строк 475–499...



Внутри чанка: 100%|██████████| 25/25 [00:28<00:00,  1.13s/it]


Чекпоинт сохранён до строки 500

✅ Собрано примеров: 1000
Преобразование в массивы...
Train size: (800, 1568), Test size: (200, 1568)
PCA (64 компоненты) на train...
Масштабирование на train...

🎉 Готово! Файлы сохранены в /kaggle/working/
Train X: (800, 96), y: [400 400]
Test X: (200, 96), y: [100 100]
